# ECOD — Credit Card + NAB (Unsupervised Distribution-Based)

**ECOD** (Empirical Cumulative Distribution-based Outlier Detection, Li & Zhao TKDE 2022) is a modern, **parameter-free** unsupervised anomaly detector that replaces Isolation Forest in our ensemble.

**Why ECOD over Isolation Forest:**
- No hyperparameters to tune (no contamination rate, no n_estimators)
- O(n·d) complexity — faster than IF's O(n·t·log(n))
- Per-dimension anomaly contribution scores — directly interpretable
- Outperforms IF and 10 other baselines across 30 benchmark datasets
- Available via `pyod` library

**Datasets:** Credit Card Fraud + NAB (58 univariate time-series)

**Role in ensemble:** Unsupervised distribution-based detector — provides a statistical perspective on anomalies that complements the supervised (XGBoost+AE), reconstruction-based (LSTM-AE), and pattern-based (Matrix Profile) detectors.


## Cell 0 — Setup

In [1]:
# Cell 0 — Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyod'])

import os, glob, json, warnings, urllib.request
import numpy as np
import pandas as pd
import joblib

from pyod.models.ecod import ECOD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve,
)

warnings.filterwarnings('ignore')
RANDOMSTATE = 42
np.random.seed(RANDOMSTATE)

# Drive paths
RUNID = 'ensemble_run_001'
DRIVEROOT = '/content/drive/MyDrive/tsad_ensemble_runs'
NOTEBOOKTAG = 'ecod'

RUNDIR = os.path.join(DRIVEROOT, RUNID, NOTEBOOKTAG)
ARTIFACTDIR = os.path.join(RUNDIR, 'artifacts')
PREDICTIONSDIR = os.path.join(RUNDIR, 'predictions')
CACHEDIR = os.path.join(DRIVEROOT, '_cache')

for d in [ARTIFACTDIR, PREDICTIONSDIR, CACHEDIR]:
    os.makedirs(d, exist_ok=True)

print('RUNDIR       :', RUNDIR)
print('ARTIFACTDIR  :', ARTIFACTDIR)
print('PREDICTIONSDIR:', PREDICTIONSDIR)
print('pyod ECOD loaded successfully')


Mounted at /content/drive
RUNDIR       : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod
ARTIFACTDIR  : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/artifacts
PREDICTIONSDIR: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions
pyod ECOD loaded successfully


## Cell 1 — Dataset Paths

In [2]:
# Cell 1 — Dataset Paths

MYDRIVE = '/content/drive/MyDrive'
CREDITCARDPATH = os.path.join(MYDRIVE, 'creditcard.csv')

NAB_CANDIDATES = [
    os.path.join(MYDRIVE, 'NAB Dataset'),
    os.path.join(MYDRIVE, 'NAB'),
    os.path.join(MYDRIVE, 'datasets', 'NAB Dataset'),
    os.path.join(MYDRIVE, 'datasets', 'NAB'),
]

def resolve_nab_root(candidates):
    for c in candidates:
        if os.path.isdir(c):
            csvs = glob.glob(os.path.join(c, '**', '*.csv'), recursive=True)
            if len(csvs) > 0:
                return c
    return None

NABROOT = resolve_nab_root(NAB_CANDIDATES)

# NAB labels
NABLABELSLOCAL = os.path.join(CACHEDIR, 'nab_combined_windows.json')
NABLABELSURL = 'https://raw.githubusercontent.com/numenta/NAB/master/labels/combined_windows.json'
if not os.path.exists(NABLABELSLOCAL):
    urllib.request.urlretrieve(NABLABELSURL, NABLABELSLOCAL)
with open(NABLABELSLOCAL) as f:
    NABWINDOWSMAP = json.load(f)

assert os.path.exists(CREDITCARDPATH), f'creditcard.csv not found at {CREDITCARDPATH}'
assert NABROOT is not None, 'NAB root not found.'

nab_csvs = [f for f in glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True) if 'README' not in f]
print(f'Credit Card : {CREDITCARDPATH}')
print(f'NAB root    : {NABROOT} ({len(nab_csvs)} CSVs)')
print(f'NAB labels  : {len(NABWINDOWSMAP)} entries')


Credit Card : /content/drive/MyDrive/creditcard.csv
NAB root    : /content/drive/MyDrive/NAB Dataset (58 CSVs)
NAB labels  : 58 entries


## Cell 2 — Shared Utilities

In [3]:
# Cell 2 — Shared Utilities

def keep_runs(y, min_len=3):
    y = np.asarray(y, dtype=np.int8).copy()
    n = len(y)
    i = 0
    while i < n:
        if y[i] == 1:
            j = i
            while j < n and y[j] == 1:
                j += 1
            if (j - i) < min_len:
                y[i:j] = 0
            i = j
        else:
            i += 1
    return y

def point_adjust(y_true, y_pred):
    yt = np.asarray(y_true, dtype=np.int8)
    yp = np.asarray(y_pred, dtype=np.int8).copy()
    n = len(yt)
    i = 0
    while i < n:
        if yt[i] == 1:
            j = i
            while j < n and yt[j] == 1:
                j += 1
            if yp[i:j].any():
                yp[i:j] = 1
            i = j
        else:
            i += 1
    return yp

def best_f1_threshold(y, scores, ngrid=300, qlo=0.50, qhi=0.999, min_run=0):
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if y.sum() == 0:
        return float(np.percentile(scores, 99.5)), 0.0
    qs = np.linspace(qlo, qhi, ngrid)
    thr_list = np.unique(np.quantile(scores, qs))
    best_t, best_f = float(thr_list[-1]), -1.0
    for t in thr_list:
        p = (scores >= t).astype(int)
        if min_run > 0:
            p = keep_runs(p, min_len=min_run)
        f = f1_score(y, p, zero_division=0)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

def compute_metrics(y, pred, scores=None, prefix=''):
    m = {
        f'{prefix}precision': float(precision_score(y, pred, zero_division=0)),
        f'{prefix}recall': float(recall_score(y, pred, zero_division=0)),
        f'{prefix}f1': float(f1_score(y, pred, zero_division=0)),
    }
    if scores is not None and len(np.unique(y)) == 2:
        m[f'{prefix}rocauc'] = float(roc_auc_score(y, scores))
        m[f'{prefix}prauc'] = float(average_precision_score(y, scores))
    else:
        m[f'{prefix}rocauc'] = float('nan')
        m[f'{prefix}prauc'] = float('nan')
    return m

print('Utilities ready.')


Utilities ready.


## Cell 3 — Credit Card ECOD

ECOD is completely unsupervised — it models the empirical CDF of each feature
on normal training data, then scores test points by how far they fall into
the distribution tails across all dimensions.

- Train on normal transactions only (semi-supervised protocol)
- Threshold tuned on validation set via PR-curve
- Evaluated on strict holdout test set (20%, same split as other models)
- Entity ID = `creditcard` for coordinator compatibility


In [4]:
# Cell 3 — Credit Card ECOD

print('Loading Credit Card data...')
df = pd.read_csv(CREDITCARDPATH).dropna().reset_index(drop=True)
y_all = df['Class'].astype(int).values

# Feature engineering (same as IF improved for consistency)
X_df = df.copy()
X_df['hoursin'] = np.sin(2 * np.pi * X_df['Time'] / 86400.0)
X_df['hourcos'] = np.cos(2 * np.pi * X_df['Time'] / 86400.0)
X_df['Amountlog'] = np.log1p(X_df['Amount'])
X_df['AmountSq'] = X_df['Amount'] ** 2
for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']:
    X_df[f'{v}_abs'] = X_df[v].abs()

feature_cols = ([f'V{i}' for i in range(1, 29)]
                + ['Amountlog', 'AmountSq', 'hoursin', 'hourcos']
                + [f'{v}_abs' for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']])

X_all = X_df[feature_cols].values.astype(np.float64)

# Same split as all other models: 80/20 stratified, RANDOM_STATE=42
idx = np.arange(len(y_all), dtype=np.int64)
idx_tune, idx_test = train_test_split(idx, test_size=0.20, stratify=y_all, random_state=RANDOMSTATE)
idx_train, idx_val = train_test_split(idx_tune, test_size=0.25, stratify=y_all[idx_tune], random_state=RANDOMSTATE)

X_train_normal = X_all[idx_train][y_all[idx_train] == 0]
X_val = X_all[idx_val]
y_val = y_all[idx_val]
X_test = X_all[idx_test]
y_test = y_all[idx_test]

print(f'Train normal: {len(X_train_normal):,} rows')
print(f'Validation  : {len(X_val):,} rows ({y_val.sum()} frauds)')
print(f'Test        : {len(X_test):,} rows ({y_test.sum()} frauds)')

# Fit ECOD on normal training data
print('Fitting ECOD (parameter-free)...')
ecod = ECOD()
ecod.fit(X_train_normal)

# Score all sets
scores_val = ecod.decision_function(X_val)
scores_test = ecod.decision_function(X_test)

# Threshold from PR-curve on validation
prec, rec, thr = precision_recall_curve(y_val, scores_val)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
if len(thr) > 0:
    best_idx = int(np.nanargmax(f1s))
    cc_threshold = float(thr[best_idx])
    print(f'Threshold from PR-curve: {cc_threshold:.4f} (val F1={f1s[best_idx]:.4f})')
else:
    cc_threshold = float(np.percentile(scores_val, 99.5))
    print(f'Fallback threshold: {cc_threshold:.4f}')

# Test predictions
cc_preds = (scores_test >= cc_threshold).astype(np.int8)
cc_scores = scores_test.astype(np.float32)
cc_ytrue = y_test.astype(np.int8)

strict = compute_metrics(cc_ytrue, cc_preds, cc_scores)

cc_results = {
    'dataset': 'creditcard',
    'protocol': 'semi-supervised strict holdout (20% test)',
    **strict,
    'threshold': cc_threshold,
    'n_anomalies_true': int(cc_ytrue.sum()),
    'n_anomalies_pred': int(cc_preds.sum()),
}

# Save artifact
joblib.dump({
    'model': ecod,
    'feature_cols': feature_cols,
    'threshold': cc_threshold,
    'test_indices': idx_test,
}, os.path.join(ARTIFACTDIR, 'ecod_creditcard.joblib'))

print()
print('=' * 60)
print('CREDIT CARD ECOD RESULTS')
print('=' * 60)
for k, v in cc_results.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')


Loading Credit Card data...
Train normal: 170,588 rows
Validation  : 56,962 rows (99 frauds)
Test        : 56,962 rows (98 frauds)
Fitting ECOD (parameter-free)...
Threshold from PR-curve: 159.7351 (val F1=0.3704)

CREDIT CARD ECOD RESULTS
  dataset                  : creditcard
  protocol                 : semi-supervised strict holdout (20% test)
  precision                : 0.342105
  recall                   : 0.530612
  f1                       : 0.416000
  rocauc                   : 0.961591
  prauc                    : 0.353613
  threshold                : 159.735134
  n_anomalies_true         : 98
  n_anomalies_pred         : 152


## Cell 4 — NAB ECOD

For NAB univariate time-series, ECOD operates on **windowed features**:
rolling statistics computed over multiple window sizes, differences, z-scores.

Protocol: train 40% / validate 30% / test 30% temporal split per series.
ECOD fitted on normal portion of training window only.
Threshold tuned on validation. Evaluated on test.
Both strict and point-adjusted metrics reported.


In [5]:
# Cell 4 — NAB ECOD

WINDOW_SIZES = [10, 30, 60]
MIN_RUN = 3
TRAINFRAC = 0.40
VALFRAC = 0.30

def extract_ts_features(values):
    # Build rich feature set from univariate time-series
    s = pd.Series(values.astype(np.float64))
    parts = [s.rename('value')]
    for w in WINDOW_SIZES:
        roll = s.rolling(w, min_periods=1)
        parts.extend([
            roll.mean().rename(f'rmean_{w}'),
            roll.std().fillna(0).rename(f'rstd_{w}'),
            roll.min().rename(f'rmin_{w}'),
            roll.max().rename(f'rmax_{w}'),
            (roll.max() - roll.min()).rename(f'rrange_{w}'),
            roll.median().rename(f'rmedian_{w}'),
        ])
    parts.extend([
        s.diff(1).fillna(0).rename('diff1'),
        s.diff(3).fillna(0).rename('diff3'),
        s.diff(10).fillna(0).rename('diff10'),
        s.pct_change().fillna(0).clip(-10, 10).rename('pct_change'),
    ])
    # Z-score relative to rolling window
    rmean = s.rolling(30, min_periods=1).mean()
    rstd = s.rolling(30, min_periods=1).std().fillna(1) + 1e-9
    parts.append(((s - rmean) / rstd).rename('zscore_30'))

    df = pd.concat(parts, axis=1)
    return np.nan_to_num(df.values, nan=0.0, posinf=0.0, neginf=0.0)

def match_nab_key(csv_path):
    rel = os.path.relpath(csv_path, NABROOT).replace('\\', '/')
    if rel in NABWINDOWSMAP:
        return rel
    base = os.path.basename(csv_path)
    for k in NABWINDOWSMAP:
        if os.path.basename(k) == base:
            return k
    return None

def make_nab_labels(timestamps, csv_path):
    y = np.zeros(len(timestamps), dtype=np.int8)
    key = match_nab_key(csv_path)
    if key and NABWINDOWSMAP.get(key):
        for t0, t1 in NABWINDOWSMAP[key]:
            y[(timestamps >= pd.Timestamp(t0)) & (timestamps <= pd.Timestamp(t1))] = 1
    return y

# Process all NAB series
all_csv = sorted([f for f in glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True)
                  if 'README' not in f])
print(f'Processing {len(all_csv)} NAB series with ECOD...')

bundle = {}
S_all, Y_all, P_all, PA_all = [], [], [], []
category_data = {}
skipped = 0

for i, csv_path in enumerate(all_csv):
    try:
        df_nab = pd.read_csv(csv_path)
        ts = pd.to_datetime(df_nab.iloc[:, 0])
        vals = df_nab.iloc[:, 1].values.astype(np.float64)
        y = make_nab_labels(ts, csv_path)
        X = extract_ts_features(vals)

        n = len(y)
        n_tr = int(round(n * TRAINFRAC))
        n_val = int(round(n * VALFRAC))
        sl_train = slice(0, n_tr)
        sl_val = slice(n_tr, n_tr + n_val)
        sl_test = slice(n_tr + n_val, n)

        if sl_test.stop <= sl_test.start or n_tr < 30:
            skipped += 1
            continue

        # Train ECOD on normal portion of training window
        X_tr = X[sl_train]
        y_tr = y[sl_train]
        X_tr_normal = X_tr[y_tr == 0] if y_tr.sum() > 0 else X_tr

        if len(X_tr_normal) < 20:
            skipped += 1
            continue

        ecod_nab = ECOD()
        ecod_nab.fit(X_tr_normal)

        # Score full series
        scores_full = ecod_nab.decision_function(X).astype(np.float32)

        # Threshold on validation
        y_v = y[sl_val]
        s_v = scores_full[sl_val]
        if y_v.sum() > 0:
            thr, val_f1 = best_f1_threshold(y_v, s_v, ngrid=300, min_run=MIN_RUN)
        else:
            thr = float(np.percentile(s_v, 99.0))
            val_f1 = 0.0

        pred_full = keep_runs((scores_full >= thr).astype(np.int8), min_len=MIN_RUN)

        # Test metrics
        y_h = y[sl_test]
        s_h = scores_full[sl_test]
        p_h = pred_full[sl_test]
        pa_h = point_adjust(y_h, p_h)

        entity_name = os.path.basename(csv_path)
        rel_key = os.path.relpath(csv_path, NABROOT).replace('\\', '/')

        bundle[entity_name] = {
            'entity_id': entity_name,
            'nab_key': rel_key,
            'scores_full': scores_full,
            'y_full': y.astype(np.int8),
            'pred_full': pred_full,
            'row_id': np.arange(len(y), dtype=np.int64),
            'val_slice': (sl_val.start, sl_val.stop),
            'test_slice': (sl_test.start, sl_test.stop),
            'threshold': float(thr),
            'val_f1': float(val_f1),
        }

        S_all.append(s_h)
        Y_all.append(y_h)
        P_all.append(p_h)
        PA_all.append(pa_h)

        cat = rel_key.split('/')[0] if '/' in rel_key else 'unknown'
        if cat not in category_data:
            category_data[cat] = {'S': [], 'Y': [], 'P': [], 'PA': []}
        category_data[cat]['S'].append(s_h)
        category_data[cat]['Y'].append(y_h)
        category_data[cat]['P'].append(p_h)
        category_data[cat]['PA'].append(pa_h)

    except Exception as e:
        print(f'[ECOD-NAB] skip {os.path.basename(csv_path)}: {e}')
        skipped += 1

print(f'Processed {len(bundle)} series, skipped {skipped}')

if S_all:
    S = np.concatenate(S_all)
    Y = np.concatenate(Y_all)
    P = np.concatenate(P_all)
    PA = np.concatenate(PA_all)

    strict = compute_metrics(Y, P, S, prefix='strict_')
    pa_m = compute_metrics(Y, PA, S, prefix='pa_')

    nab_results = {
        'dataset': 'nab',
        'protocol': f'temporal holdout: train {int(100*TRAINFRAC)}% / val {int(100*VALFRAC)}% / test {int(100*(1-TRAINFRAC-VALFRAC))}%',
        **strict,
        **pa_m,
        'n_anomalies_true': int(Y.sum()),
        'n_anomalies_pred': int(P.sum()),
        'n_series': len(bundle),
    }
else:
    nab_results = {'dataset': 'nab', 'strict_f1': 0.0, 'n_series': 0}

print()
print('=' * 60)
print('NAB ECOD RESULTS (GLOBAL)')
print('=' * 60)
for k, v in nab_results.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')


Processing 58 NAB series with ECOD...
Processed 58 series, skipped 0

NAB ECOD RESULTS (GLOBAL)
  dataset                  : nab
  protocol                 : temporal holdout: train 40% / val 30% / test 30%
  strict_precision         : 0.163236
  strict_recall            : 0.320213
  strict_f1                : 0.216239
  strict_rocauc            : 0.685096
  strict_prauc             : 0.291914
  pa_precision             : 0.358178
  pa_recall                : 0.916032
  pa_f1                    : 0.514990
  pa_rocauc                : 0.685096
  pa_prauc                 : 0.291914
  n_anomalies_true         : 11814
  n_anomalies_pred         : 23175
  n_series                 : 58


## Cell 5 — NAB Per-Category Breakdown

In [6]:
# Cell 5 — NAB Per-Category Breakdown

print('NAB ECOD PER-CATEGORY RESULTS')
print('=' * 90)

cat_rows = []
for cat, data in sorted(category_data.items()):
    Y_c = np.concatenate(data['Y'])
    P_c = np.concatenate(data['P'])
    S_c = np.concatenate(data['S'])
    PA_c = np.concatenate(data['PA'])

    strict = compute_metrics(Y_c, P_c, S_c)
    pa = compute_metrics(Y_c, PA_c, S_c, prefix='pa_')

    row = {
        'category': cat,
        'n_series': len(data['Y']),
        'n_points': len(Y_c),
        'n_anomalies': int(Y_c.sum()),
        'anomaly_rate': float(Y_c.mean()),
        **strict,
        **pa,
    }
    cat_rows.append(row)
    print(f"  {cat:30s}  series={row['n_series']:3d}  "
          f"strict F1={strict['f1']:.4f}  PA F1={pa['pa_f1']:.4f}  "
          f"anomaly_rate={row['anomaly_rate']:.4f}")

nab_cat_df = pd.DataFrame(cat_rows)
display(nab_cat_df)


NAB ECOD PER-CATEGORY RESULTS
  artificialNoAnomaly             series=  5  strict F1=0.0000  PA F1=0.0000  anomaly_rate=0.0000
  artificialWithAnomaly           series=  6  strict F1=0.3040  PA F1=0.5780  anomaly_rate=0.2308
  realAWSCloudwatch               series= 17  strict F1=0.1397  PA F1=0.3652  anomaly_rate=0.0943
  realAdExchange                  series=  6  strict F1=0.2896  PA F1=0.3719  anomaly_rate=0.0898
  realKnownCause                  series=  7  strict F1=0.3510  PA F1=0.7688  anomaly_rate=0.1873
  realTraffic                     series=  7  strict F1=0.2688  PA F1=0.7027  anomaly_rate=0.2160
  realTweets                      series= 10  strict F1=0.1481  PA F1=0.4130  anomaly_rate=0.0639


,category,n_series,n_points,n_anomalies,anomaly_rate,precision,recall,f1,rocauc,prauc,pa_precision,pa_recall,pa_f1,pa_rocauc,pa_prauc
0,artificialNoAnomaly,5,6045,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN
1,artificialWithAnomaly,6,7254,1674,0.230769,0.256684,0.372760,0.304019,0.649444,0.412156,0.439168,0.845281,0.578023,0.649444,0.412156
2,realAWSCloudwatch,17,20315,1916,0.094315,0.093371,0.277140,0.139682,0.606176,0.242421,0.234561,0.824635,0.365233,0.606176,0.242421
3,realAdExchange,6,2884,259,0.089806,0.179944,0.741313,0.289593,0.626783,0.227539,0.228395,1.000000,0.371859,0.626783,0.227539
4,realKnownCause,7,20868,3908,0.187272,0.361760,0.340839,0.350988,0.835985,0.593362,0.624481,1.000000,0.768837,0.835985,0.593362
5,realTraffic,7,4699,1015,0.216003,0.253043,0.286700,0.268822,0.690076,0.457378,0.541622,1.000000,0.702665,0.690076,0.457378
6,realTweets,10,47590,3042,0.063921,0.102470,0.267258,0.148141,0.531957,0.100629,0.270838,0.869494,0.413023,0.531957,0.100629


## Cell 6 — Export for Coordinator

In [7]:
# Cell 6 — Export for Coordinator

exported = {}
summaryrows = []

# --- Credit Card ---
cc_bundle = {
    'dataset': 'creditcard',
    'model': 'ecod',
    'protocol': 'semi-supervised strict holdout',
    'entities': {
        'creditcard': {
            'entityid': 'creditcard',
            'scoresfull': cc_scores,
            'yfull': cc_ytrue,
            'predfull': cc_preds,
            'rowid': np.arange(len(cc_ytrue), dtype=np.int64),
            'originalrowid': idx_test.astype(np.int64),
            'threshold': cc_threshold,
        }
    }
}

ccpath = os.path.join(PREDICTIONSDIR, 'ecod_creditcard_strict.joblib')
joblib.dump(cc_bundle, ccpath)
exported['creditcard'] = ccpath
summaryrows.append({'dataset': 'creditcard', **{k: v for k, v in cc_results.items() if k != 'dataset'}})
print('Saved', ccpath)

# --- NAB ---
nab_payload = {
    'dataset': 'nab',
    'model': 'ecod',
    'protocol': nab_results.get('protocol', 'temporal holdout'),
    'entities': bundle,
}
nabpath = os.path.join(PREDICTIONSDIR, 'ecod_nab_strict.joblib')
joblib.dump(nab_payload, nabpath)
exported['nab'] = nabpath
summaryrows.append({'dataset': 'nab', **{k: v for k, v in nab_results.items() if k != 'dataset'}})
print('Saved', nabpath)

# --- Summary ---
summary = pd.DataFrame(summaryrows)
summarypath = os.path.join(PREDICTIONSDIR, 'ecod_summary.csv')
summary.to_csv(summarypath, index=False)
print('Saved', summarypath)
display(summary)

# --- Manifest ---
manifest = {
    'runid': RUNID,
    'driveroot': DRIVEROOT,
    'notebooktag': NOTEBOOKTAG,
    'modelfamily': 'ecod',
    'exportprotocol': 'ensembleexportv2',
    'artifactsdir': ARTIFACTDIR,
    'predictionsdir': PREDICTIONSDIR,
    'exports': exported,
    'summarycsv': summarypath,
    'creditcard_originalrowid_included': True,
    'datasets': ['creditcard', 'nab'],
}

manifestpath = os.path.join(PREDICTIONSDIR, 'ecod_manifest.json')
with open(manifestpath, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Saved', manifestpath)

print()
print('Coordinator-ready files in:', PREDICTIONSDIR)


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions/ecod_creditcard_strict.joblib
Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions/ecod_nab_strict.joblib
Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions/ecod_summary.csv


,dataset,protocol,precision,recall,f1,rocauc,prauc,threshold,n_anomalies_true,n_anomalies_pred,...,strict_recall,strict_f1,strict_rocauc,strict_prauc,pa_precision,pa_recall,pa_f1,pa_rocauc,pa_prauc,n_series
0,creditcard,semi-supervised strict holdout (20% test),0.342105,0.530612,0.416,0.961591,0.353613,159.735134,98,152,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,nab,temporal holdout: train 40% / val 30% / test 30%,NaN,NaN,NaN,NaN,NaN,NaN,11814,23175,...,0.320213,0.216239,0.685096,0.291914,0.358178,0.916032,0.51499,0.685096,0.291914,58.0


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions/ecod_manifest.json

Coordinator-ready files in: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions


## Cell 7 — Final Summary

In [8]:
# Cell 7 — Final Summary

print('=' * 70)
print('ECOD FINAL SUMMARY')
print('=' * 70)

print()
print('CREDIT CARD:')
for k, v in cc_results.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

print()
print('NAB (global):')
for k, v in nab_results.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

print()
print('NAB (per-category):')
display(nab_cat_df[['category', 'n_series', 'n_anomalies', 'f1', 'pa_f1', 'rocauc']].round(4))

print()
print('All exports saved to:', PREDICTIONSDIR)
print('Done.')


ECOD FINAL SUMMARY

CREDIT CARD:
  dataset                  : creditcard
  protocol                 : semi-supervised strict holdout (20% test)
  precision                : 0.342105
  recall                   : 0.530612
  f1                       : 0.416000
  rocauc                   : 0.961591
  prauc                    : 0.353613
  threshold                : 159.735134
  n_anomalies_true         : 98
  n_anomalies_pred         : 152

NAB (global):
  dataset                  : nab
  protocol                 : temporal holdout: train 40% / val 30% / test 30%
  strict_precision         : 0.163236
  strict_recall            : 0.320213
  strict_f1                : 0.216239
  strict_rocauc            : 0.685096
  strict_prauc             : 0.291914
  pa_precision             : 0.358178
  pa_recall                : 0.916032
  pa_f1                    : 0.514990
  pa_rocauc                : 0.685096
  pa_prauc                 : 0.291914
  n_anomalies_true         : 11814
  n_anomalies_pred  

,category,n_series,n_anomalies,f1,pa_f1,rocauc
0,artificialNoAnomaly,5,0,0.0000,0.0000,NaN
1,artificialWithAnomaly,6,1674,0.3040,0.5780,0.6494
2,realAWSCloudwatch,17,1916,0.1397,0.3652,0.6062
3,realAdExchange,6,259,0.2896,0.3719,0.6268
4,realKnownCause,7,3908,0.3510,0.7688,0.8360
5,realTraffic,7,1015,0.2688,0.7027,0.6901
6,realTweets,10,3042,0.1481,0.4130,0.5320



All exports saved to: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions
Done.


In [9]:
# DEBUG — ECOD performance analysis
print('=== ECOD DIAGNOSTIC ANALYSIS ===')
print()
print('ECOD STRENGTHS in this ensemble:')
print('  - Parameter-free: no contamination rate or n_estimators to tune')
print('  - High ROC-AUC (0.96): good at RANKING anomalies even if F1 is low')
print('  - Different perspective: distribution-based vs tree-based/reconstruction-based')
print('  - Strong on NAB realKnownCause category (PA F1=0.77)')
print()
print('ECOD LIMITATIONS on Credit Card:')
print('  - Low precision (0.34): too many false positives')
print('  - ECOD assumes each feature is independently distributed')
print('  - Credit card fraud involves feature INTERACTIONS (V14*V17, etc.)')
print('  - This independence assumption limits CC precision')
print()
print('ROLE IN ENSEMBLE:')
print('  - CC: low weight in fusion (F1^2 weighting downweights automatically)')
print('  - NAB: valuable for diversity — catches different anomaly patterns')
print('  - Provides unsupervised baseline that complements supervised models')
print()
print('Published justification: Li & Zhao, TKDE 2022 — ECOD outperforms IF')
print('and 10 other baselines on 30 benchmark datasets.')


=== ECOD DIAGNOSTIC ANALYSIS ===

ECOD STRENGTHS in this ensemble:
  - Parameter-free: no contamination rate or n_estimators to tune
  - High ROC-AUC (0.96): good at RANKING anomalies even if F1 is low
  - Different perspective: distribution-based vs tree-based/reconstruction-based
  - Strong on NAB realKnownCause category (PA F1=0.77)

ECOD LIMITATIONS on Credit Card:
  - Low precision (0.34): too many false positives
  - ECOD assumes each feature is independently distributed
  - Credit card fraud involves feature INTERACTIONS (V14*V17, etc.)
  - This independence assumption limits CC precision

ROLE IN ENSEMBLE:
  - CC: low weight in fusion (F1^2 weighting downweights automatically)
  - NAB: valuable for diversity — catches different anomaly patterns
  - Provides unsupervised baseline that complements supervised models

Published justification: Li & Zhao, TKDE 2022 — ECOD outperforms IF
and 10 other baselines on 30 benchmark datasets.
